# GoEmotions Dataset

In [ ]:


import pandas as pd
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

base_url = "hf://datasets/google-research-datasets/go_emotions/simplified/"

train_df = pd.read_parquet(base_url + "train-00000-of-00001.parquet")
val_df = pd.read_parquet(base_url + "validation-00000-of-00001.parquet")
test_df = pd.read_parquet(base_url + "test-00000-of-00001.parquet")

train_df


text labels       id
0      My favourite food is anything I didn't have to...   [27]  eebbqej
1      Now if he does off himself, everyone will thin...   [27]  ed00q6i
2                         WHY THE FUCK IS BAYLESS ISOING    [2]  eezlygj
3                            To make her feel threatened   [14]  ed7ypvh
4                                 Dirty Southern Wankers    [3]  ed0bdzj
...                                                  ...    ...      ...
43405  Added you mate well I’ve just got the bow and ...   [18]  edsb738
43406  Always thought that was funny but is it a refe...    [6]  ee7fdou
43407  What are you talking about? Anything bad that ...    [3]  efgbhks
43408            More like a baptism, with sexy results!   [13]  ed1naf8
43409                                    Enjoy the ride!   [17]  eecwmbq

[43410 rows x 3 columns]

## Preprocessing dataset


## Dataset Filtering and cleaning 

In [ ]:
# check size of each dataset split
print(len(train_df), len(val_df), len(test_df))

# check for nulls/empty text 
print(train_df['text'].isnull().sum())
print((train_df['text'].str.strip() == '').sum())

# check for duplicate text across splits
train_texts = set(train_df['text'])
test_texts = set(test_df['text'])
print(len(train_texts & test_texts)) 

# check label distribution
print(train_df['labels'].explode().value_counts())

43410 5426 5427
0
0
32
labels
27    14219
0      4130
4      2939
15     2662
3      2470
1      2328
7      2191
18     2086
10     2022
20     1581
2      1567
17     1452
6      1368
25     1326
9      1269
22     1110
5      1087
26     1060
13      853
11      793
8       641
14      596
24      545
12      303
19      164
23      153
21      111
16       77
Name: count, dtype: int64


In [ ]:

# Label and group mappings 
LABEL_NAMES = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]
# Map each label to its corresponding group
GROUP_MAP = {
    "admiration":"positive","amusement":"positive","approval":"positive",
    "caring":"positive","desire":"positive","excitement":"positive",
    "gratitude":"positive","joy":"positive","love":"positive",
    "optimism":"positive","pride":"positive","relief":"positive",
    "anger":"negative","annoyance":"negative","disappointment":"negative",
    "disapproval":"negative","disgust":"negative","embarrassment":"negative",
    "fear":"negative","grief":"negative","nervousness":"negative",
    "remorse":"negative","sadness":"negative",
    "confusion":"ambiguous","curiosity":"ambiguous",
    "realization":"ambiguous","surprise":"ambiguous",
    "neutral":"neutral"
}
# Function to convert label IDs to their corresponding groups
def labels_to_groups_all(label_ids):
    names = [LABEL_NAMES[i] for i in label_ids]
    groups = [GROUP_MAP[n] for n in names]
    return list(dict.fromkeys(groups))

# Process each dataset split
def process_split(df):
    df = df.copy()
    df["all_groups"] = df["labels"].apply(labels_to_groups_all)
    df["num_distinct_groups"] = df["all_groups"].apply(len)
    clean = df[df["num_distinct_groups"] == 1].copy()
    clean["group_label"] = clean["all_groups"].apply(lambda g: g[0])
    conflict = df[df["num_distinct_groups"] > 1].copy()
    return clean, conflict

# Process all dataset splits
train_clean, train_conflict = process_split(train_df)
val_clean, val_conflict = process_split(val_df)
test_clean, test_conflict = process_split(test_df)

In [ ]:
# Display statistics for each dataset split
print(f"Train: {len(train_clean)} clean / {len(train_conflict)} conflicting "
      f"({len(train_conflict)/len(train_df)*100:.1f}%)")
print(f"Val:   {len(val_clean)} clean / {len(val_conflict)} conflicting "
      f"({len(val_conflict)/len(val_df)*100:.1f}%)")
print(f"Test:  {len(test_clean)} clean / {len(test_conflict)} conflicting "
      f"({len(test_conflict)/len(test_df)*100:.1f}%)")

# Display label distribution for the clean datasets
print(train_clean["group_label"].value_counts())
print(val_clean["group_label"].value_counts())
print(test_clean["group_label"].value_counts())

Train: 40030 clean / 3380 conflicting (7.8%)
Val:   5006 clean / 420 conflicting (7.7%)
Test:  5027 clean / 400 conflicting (7.4%)
group_label
positive     15216
neutral      12823
negative      8133
ambiguous     3858
Name: count, dtype: int64
group_label
positive     1941
neutral      1592
negative     1014
ambiguous     459
Name: count, dtype: int64
group_label
positive     1863
neutral      1606
negative     1070
ambiguous     488
Name: count, dtype: int64


In [ ]:
# Count how many rows in the training set have more than one label
multi_label_count = (train_df["labels"].apply(len) > 1).sum()
print(multi_label_count, "of", len(train_df), "rows have more than one label")
print(multi_label_count / len(train_df) * 100, "%")

7102 of 43410 rows have more than one label
16.360285648468096 %


## Data Visualization